# Repeated replies: is the signal just catchphrases?

The signal (Perplexity.ipynb) is 19.8 percentage points: perplexity drops more on my test replies than on my friends'. A reviewer's objection: many of my replies are stock phrases that repeat in train (9.0% of my test replies are exact copies of one of my train messages; StyleClassifier.ipynb), and a fine-tuned model predicts those cheaply. If the signal were only that, it would vanish once repeated replies are removed.

`scripts/eval_repeats.py` recomputes the signal of the final adapter after removing, on each side symmetrically, the test replies that repeat that speaker's own train replies: exact repeats (lowercased, spaces collapsed), and replies among that speaker's 100 most frequent train replies or of 5 characters or fewer. No model: it reuses the per-reply numbers of `signal_run1_5ep.json`. Intervals by session and by chat (ByChat.ipynb). Texts are compared, never printed.

## Prediction (written before running)

Fede: it will go down, depending on how many repetitions there are: if 20% of the replies are removed, it drops about that much. But it survives.

Claude: it goes down somewhat but survives. Length.ipynb put most of the signal in replies of 8 tokens or more (17 to 18.5 points), and catchphrases are mostly short replies. If it fell to 0, "it learned me" would really be "it learned my stock phrases".

How it's read: the signal survives if its interval (by chat) still excludes 0 after removing both kinds of repeats.

In [1]:
import subprocess
from pathlib import Path
ROOT = Path("..").resolve()
p = subprocess.run("HF_HUB_DISABLE_PROGRESS_BARS=1 uv run python scripts/eval_repeats.py --out experiments/results/repeats.json",
                   shell=True, cwd=ROOT, capture_output=True, text=True)
print("\n".join(l for l in p.stdout.splitlines() if "Warning" not in l))
assert p.returncode == 0, p.stderr[-2000:]

all replies                      kept: mine 100.0%, friends 100.0% | signal 19.8 points | by session [14.7, 25.8] | by chat [8.9, 26.0]
without exact repeats            kept: mine 89.8%, friends 92.3% | signal 19.0 points | by session [13.6, 24.6] | by chat [8.6, 24.5]
without frequent or very short   kept: mine 90.0%, friends 90.4% | signal 19.8 points | by session [14.4, 25.3] | by chat [9.3, 25.2]
without either                   kept: mine 87.6%, friends 88.9% | signal 19.6 points | by session [14.2, 25.1] | by chat [9.3, 24.8]


## Conclusion

**Prediction:** Fede, a drop about as large as the share of replies removed, but it survives; Claude, it goes down somewhat and survives. **Both right that it survives; it barely moves, so both predicted more of a drop than there is.**

| Replies kept | Mine | Friends' | Signal | By session | By chat |
|---|---|---|---|---|---|
| all | 100% | 100% | 19.8 | [14.7, 25.8] | [8.9, 26.0] |
| without exact repeats of own train replies | 89.8% | 92.3% | 19.0 | [13.6, 24.6] | [8.6, 24.5] |
| without the 100 most frequent or ≤5 characters | 90.0% | 90.4% | 19.8 | [14.4, 25.3] | [9.3, 25.2] |
| without either | 87.6% | 88.9% | 19.6 | [14.2, 25.1] | [9.3, 24.8] |

Removing about 12% of the replies on each side changes the signal by 0.2 to 0.8 points. The signal is not in the repeated replies: those are easy for the adapter on *both* sides (my friends repeat themselves too), so they add to both drops about equally and cancel in the difference. What makes my drop larger is in the replies that don't repeat, which fits Length.ipynb: the gap is where there is something to say.

**Limits.** Exact and frequent repeats only; a paraphrase of a stock phrase is not caught. My friends are pooled into one group, so "their 100 most frequent replies" is a group's list, not one person's. This rules out catchphrases as the explanation, not the asymmetry of the control (the adapter saw only me, under a prompt that names me): that is CrossSpeaker.ipynb.